# 📚 Slide Mentor — a simple RAG chatbot (single notebook)

Upload lecture-slide PDFs and chat with them. The notebook extracts the text, splits it into
chunks, embeds the chunks locally (sentence-transformers), stores them in ChromaDB, and uses the
Groq API to answer questions from the most relevant slides, with page sources.

**Pipeline**

`PDF → text → clean → chunks → embeddings → ChromaDB`
`Question → embedding → similarity search → top chunks → LLM → answer`

This combines `rag.py` and `app.py` from the original project. The Streamlit UI is replaced with
an `ipywidgets` interface so everything runs inside Jupyter / Colab.

**Run the cells top to bottom.** You need a [Groq API key](https://console.groq.com/keys).

## 1. Install dependencies

In [ ]:
%pip install -q pymupdf chromadb sentence-transformers groq python-dotenv ipywidgets

## 2. Imports, settings and API key

In [ ]:
import os
import re
from getpass import getpass

import chromadb
import fitz  # PyMuPDF
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()  # reads GROQ_API_KEY from a .env file (if there is one)

# ---------- Settings ----------
CHROMA_PATH = "./chroma_db"          # folder where the vector DB is saved
COLLECTION_NAME = "slide_documents"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "openai/gpt-oss-120b"    # Groq model; change if you want another
CHUNK_SIZE = 800                     # characters per chunk
CHUNK_OVERLAP = 100                  # characters shared between neighbouring chunks
TOP_K = 5                            # how many chunks we retrieve per question
NOT_FOUND = "I couldn't find this information in the uploaded slides."

# ---------- API key ----------
# Order: existing env var / .env file -> Colab secret -> prompt
if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata  # only exists on Google Colab
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 3. Load the embedding model and the vector database

The first run downloads the embedding model (~90 MB). Processed slides are saved in `./chroma_db`,
so they are still there if you restart the kernel. Delete that folder to start fresh.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)  # saved on disk
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # compare vectors by cosine distance
)
print("Chunks currently stored:", collection.count())

## 4. The RAG pipeline

In [ ]:
# ---------- 1. Extract text ----------
def extract_text_from_pdf(pdf_bytes):
    """Return a list of (page_number, text) for every page in the PDF."""
    pages = []
    with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
        for page_number, page in enumerate(pdf, start=1):
            pages.append((page_number, page.get_text()))
    return pages


# ---------- 2. Clean text ----------
def clean_text(text):
    """Collapse all whitespace/newlines into single spaces."""
    return re.sub(r"\s+", " ", text).strip()


# ---------- 3. Chunking ----------
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into pieces of `chunk_size` characters that overlap a little."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap  # step back a bit so chunks overlap
    return chunks


# ---------- 4. Embed + store ----------
def add_pdf_to_vector_db(filename, pdf_bytes, on_step=print):
    """Run the full ingestion pipeline for one PDF. Returns the number of chunks stored."""
    on_step("Reading PDF...")
    pages = extract_text_from_pdf(pdf_bytes)

    on_step("Creating chunks...")
    ids, texts, metadatas = [], [], []
    for page_number, raw_text in pages:
        text = clean_text(raw_text)
        for i, chunk in enumerate(chunk_text(text)):
            ids.append(f"{filename}_p{page_number}_c{i}")
            texts.append(chunk)
            metadatas.append({"source": filename, "page": page_number})

    if not texts:
        return 0  # e.g. a scanned PDF with no selectable text

    on_step("Generating embeddings...")
    embeddings = embedding_model.encode(texts).tolist()

    on_step("Storing in vector database...")
    collection.delete(where={"source": filename})  # avoid duplicates if re-uploaded
    collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)
    return len(texts)


def list_documents():
    """Names of all PDFs currently stored in the vector DB."""
    if collection.count() == 0:
        return []
    metadatas = collection.get(include=["metadatas"])["metadatas"]
    return sorted({m["source"] for m in metadatas})


# ---------- 5. Retrieval ----------
def retrieve_context(question, top_k=TOP_K):
    """Embed the question and return the most similar chunks from ChromaDB."""
    if collection.count() == 0:
        return []
    question_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=min(top_k, collection.count()),
    )
    chunks = []
    for text, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append({"text": text, "source": meta["source"], "page": meta["page"]})
    return chunks


# ---------- 6. Generation ----------
def generate_answer(question, chunks):
    """Put the retrieved chunks + question into a prompt and ask the Groq LLM."""
    context = "\n\n".join(
        f"[{c['source']}, page {c['page']}]\n{c['text']}" for c in chunks
    )
    prompt = f"""You are a helpful teaching assistant.

Answer the user's question using ONLY the provided context from the uploaded slides.

If the answer is not present in the context, reply exactly:
"{NOT_FOUND}"

Explain concepts clearly and simply.

Context:
{context}

Question:
{question}"""

    client = Groq()  # reads GROQ_API_KEY from the environment
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


def ask_question(question):
    """Full query pipeline. Returns (answer, sources) where sources is a list of strings."""
    chunks = retrieve_context(question)
    if not chunks:
        return "Please upload and process some slides first.", []

    answer = generate_answer(question, chunks)

    if NOT_FOUND in answer:  # don't show sources for an "I don't know" answer
        return answer, []
    sources = sorted({f"{c['source']} — Page {c['page']}" for c in chunks})
    return answer, sources

## 5. Chat interface

Upload one or more PDFs, click **Process Slides**, then ask questions.

*(Alternative without the UI: skip the upload widget and call
`add_pdf_to_vector_db("lecture1.pdf", open("lecture1.pdf", "rb").read())`, then
`ask_question("...")` directly.)*

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

chat_history = []  # list of {"role", "content", "sources"} dicts

# ---------- Widgets ----------
uploader = widgets.FileUpload(accept=".pdf", multiple=True, description="Upload PDF")
process_btn = widgets.Button(description="Process Slides", button_style="primary", icon="cogs")
clear_btn = widgets.Button(description="Clear Chat", icon="trash")
docs_out = widgets.Output()
status_out = widgets.Output()
chat_out = widgets.Output()
question_box = widgets.Text(
    placeholder="Ask a question about your slides...",
    layout=widgets.Layout(width="75%"),
)
ask_btn = widgets.Button(description="Ask", button_style="success", icon="paper-plane")


def get_uploaded_files():
    """Return [(name, bytes)] for the uploader (works with ipywidgets 7 and 8)."""
    value = uploader.value
    items = value.values() if isinstance(value, dict) else value
    files = []
    for f in items:
        name = f["name"]
        content = bytes(f["content"])
        files.append((name, content))
    return files


def refresh_documents():
    with docs_out:
        clear_output()
        documents = list_documents()
        if documents:
            display(Markdown("**Documents**\n" + "\n".join(f"- {n}" for n in documents)))
        else:
            display(Markdown("*No documents yet.*"))


def render_chat():
    with chat_out:
        clear_output()
        for m in chat_history:
            who = "🧑 **You**" if m["role"] == "user" else "📚 **Slide Mentor**"
            display(Markdown(f"{who}\n\n{m['content']}"))
            if m.get("sources"):
                display(Markdown("*Sources:* " + " · ".join(m["sources"])))
            display(Markdown("---"))


def on_process(_):
    files = get_uploaded_files()
    with status_out:
        clear_output()
        if not files:
            print("Please choose at least one PDF first.")
            return
        print("Processing slides...")
        for name, data in files:
            print(f"\n{name}")
            n = add_pdf_to_vector_db(name, data, on_step=lambda s: print("  " + s))
            if n == 0:
                print("  ⚠ No selectable text found (is it a scanned PDF?)")
            else:
                print(f"  ✓ {n} chunks stored")
        print("\n✓ Slides processed successfully!")
    refresh_documents()


def on_ask(_):
    question = question_box.value.strip()
    if not question:
        return
    if not os.getenv("GROQ_API_KEY"):
        with status_out:
            clear_output()
            print("GROQ_API_KEY is not set. Re-run the API key cell.")
        return
    question_box.value = ""
    chat_history.append({"role": "user", "content": question})
    ask_btn.disabled = True
    ask_btn.description = "Thinking..."
    render_chat()
    try:
        answer, sources = ask_question(question)
    except Exception as e:
        answer, sources = f"⚠ Error: {e}", []
    chat_history.append({"role": "assistant", "content": answer, "sources": sources})
    ask_btn.disabled = False
    ask_btn.description = "Ask"
    render_chat()


def on_clear(_):
    chat_history.clear()
    render_chat()


process_btn.on_click(on_process)
ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
question_box.on_submit(on_ask) if hasattr(question_box, "on_submit") else None

# ---------- Layout ----------
display(Markdown("### 📚 Slide Mentor\n*Chat with your lecture slides*"))
display(widgets.HBox([uploader, process_btn, clear_btn]))
display(status_out)
display(docs_out)
display(chat_out)
display(widgets.HBox([question_box, ask_btn]))
refresh_documents()
if not list_documents():
    print("👆 Upload a PDF and click 'Process Slides' to get started.")